# NovaHiring — Demo de Entrevistas

Este notebook simula 4 conversaciones de entrevista con diferentes perfiles de candidatos y demuestra cómo el sistema:

1. Hace 8 preguntas (una por cada dimensión del scorecard)
2. Recibe respuestas de distinta calidad
3. Calcula el score ponderado usando la fórmula `Σ(score × peso) / Σ(pesos)`
4. Produce un ranking y selecciona al ganador

**Los scores de IA son pre-simulados** (no se llama a la API real). En producción cada respuesta es evaluada por `gpt-4o-mini` contra la rúbrica de su dimensión.

In [ ]:
import json
import sys
from decimal import Decimal
from pathlib import Path

# Añade el directorio raíz al path para importar los módulos del proyecto
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from contracts import DiscoveryJSON, DimensionScore
from services.scorer import Scorer
from services.interview_conductor import INTERVIEW_QUESTIONS

CONVERSATIONS_DIR = ROOT / 'docs' / 'data' / 'conversations'
SCENARIOS = ['ganador_claro', 'buen_candidato', 'candidato_promedio', 'respuestas_debiles']

print('✓ Módulos cargados correctamente')
print(f'✓ Directorio de conversaciones: {CONVERSATIONS_DIR}')

In [ ]:
# Fixture de discovery (mismo que los tests)
DISCOVERY = DiscoveryJSON(
    cliente={}, problema_negocio={}, producto_a_construir={},
    contexto_equipo={}, restricciones={},
    perfil_candidato={
        'habilidades_tecnicas': {'obligatorias': [], 'deseables': []},
        'habilidades_blandas': {}, 'senales_de_seniority': {},
        'criterios_de_descarte': [
            {'id': 'KO1', 'descripcion': 'WhatsApp', 'razon': 'x'},
            {'id': 'KO2', 'descripcion': 'RGPD', 'razon': 'x'},
            {'id': 'KO3', 'descripcion': 'Autonomy', 'razon': 'x'},
        ],
    },
    scorecard={
        'peso_total': 19,
        'dimensiones': [
            {'id': 'D1', 'nombre': 'Integraciones técnicas', 'peso': 3, 'rubricas': {'5': 'a', '3': 'b', '1': 'c'}},
            {'id': 'D2', 'nombre': 'RGPD/LOPDGDD', 'peso': 3, 'rubricas': {'5': 'a', '3': 'b', '1': 'c'}},
            {'id': 'D3', 'nombre': 'Autonomía', 'peso': 3, 'rubricas': {'5': 'a', '3': 'b', '1': 'c'}},
            {'id': 'D4', 'nombre': 'Build vs buy', 'peso': 3, 'rubricas': {'5': 'a', '3': 'b', '1': 'c'}},
            {'id': 'D5', 'nombre': 'Entrega en plazo', 'peso': 2, 'rubricas': {'5': 'a', '3': 'b', '1': 'c'}},
            {'id': 'D6', 'nombre': 'Comunicación', 'peso': 2, 'rubricas': {'5': 'a', '3': 'b', '1': 'c'}},
            {'id': 'D7', 'nombre': 'Sector salud', 'peso': 2, 'rubricas': {'5': 'a', '3': 'b', '1': 'c'}},
            {'id': 'D8', 'nombre': 'Stack', 'peso': 1, 'rubricas': {'5': 'a', '3': 'b', '1': 'c'}},
        ],
    },
    criterios_de_exito=[],
)
PESOS = {d.id: d.peso for d in DISCOVERY.dimensions}
print('✓ Scorecard cargado. Dimensiones:', [d.id for d in DISCOVERY.dimensions])
print(f'✓ Peso total: {DISCOVERY.total_weight}')

In [ ]:
def load_scenario(name):
    return json.loads((CONVERSATIONS_DIR / f'{name}.json').read_text(encoding='utf-8'))

def compute_scores(scenario):
    scores = [
        DimensionScore(
            dimension_id=a['dimension_id'],
            peso=PESOS[a['dimension_id']],
            score=Decimal(str(a['simulated_ai_score'])),
            justificacion=a['simulated_ai_justificacion'],
            evidencia=a['simulated_ai_evidencia'],
        )
        for a in scenario['answers']
    ]
    return Scorer().calculate(scores, DISCOVERY)

scenarios = {name: load_scenario(name) for name in SCENARIOS}
print(f'✓ {len(scenarios)} escenarios cargados:', list(scenarios.keys()))

## Conversaciones — Q&A por escenario

Cada escenario tiene 8 preguntas (D1–D8) con respuestas de distinta calidad.

In [ ]:
SCORE_EMOJI = {5: '🟢', 4: '🟡', 3: '🟠', 2: '🔴', 1: '⛔'}

def print_conversation(name):
    s = scenarios[name]
    print(f"\n{'='*70}")
    print(f"ESCENARIO: {s['scenario'].upper().replace('_', ' ')}")
    print(f"Candidato: {s['candidate_name']}")
    print(f"Descripción: {s['description']}")
    print(f"{'='*70}")
    for a in s['answers']:
        score = a['simulated_ai_score']
        emoji = SCORE_EMOJI[score]
        print(f"\n{emoji} [{a['dimension_id']}] {a['dimension_name']} (peso×{PESOS[a['dimension_id']]}) → score {score}/5")
        print(f"   P: {a['question'][:90]}..." if len(a['question']) > 90 else f"   P: {a['question']}")
        print(f"   R: {a['answer'][:120]}..." if len(a['answer']) > 120 else f"   R: {a['answer']}")
        print(f"   IA: {a['simulated_ai_justificacion'][:100]}..." if len(a['simulated_ai_justificacion']) > 100 else f"   IA: {a['simulated_ai_justificacion']}")

print_conversation('ganador_claro')

In [ ]:
print_conversation('buen_candidato')

In [ ]:
print_conversation('candidato_promedio')

In [ ]:
print_conversation('respuestas_debiles')

## Cálculo de scores — Fórmula: Σ(score × peso) / Σ(pesos)

In [ ]:
results = []
print(f"{'Escenario':<25} {'Candidato':<22} {'Cálculo detallado':<40} {'Score':>6} {'%':>7}")
print('-' * 105)

for name in SCENARIOS:
    s = scenarios[name]
    weighted, normalized = compute_scores(s)
    
    # Detalle del cálculo
    parts = []
    total_num = Decimal(0)
    for a in s['answers']:
        sc = a['simulated_ai_score']
        p = PESOS[a['dimension_id']]
        total_num += sc * p
        parts.append(f"{sc}×{p}")
    detalle = f"({' + '.join(parts)}) / 19 = {int(total_num)}/19"
    
    results.append({
        'name': name,
        'candidate': s['candidate_name'],
        'weighted': weighted,
        'normalized': normalized,
        'resultado': s['expected_result']['resultado'],
        'expected_pos': s['expected_result']['ranking_position'],
        'detail': detalle,
    })
    
    print(f"{name:<25} {s['candidate_name']:<22} {detalle:<40} {float(weighted):>6.2f} {float(normalized)*100:>6.1f}%")

print(f"\n✓ Scorer aplicado correctamente a {len(results)} escenarios")

## Ranking final

In [ ]:
ranked = sorted(results, key=lambda x: x['weighted'], reverse=True)

print(f"\n{'='*70}")
print("RANKING FINAL — Clínica Salud Valencia S.L.")
print(f"{'='*70}")
print(f"{'Pos':<4} {'Candidato':<22} {'Escenario':<25} {'Score /5':>8} {'Porcentaje':>11} {'Resultado':<12}")
print('-' * 84)

MEDALS = {1: '🥇', 2: '🥈', 3: '🥉', 4: '  4'}

for pos, r in enumerate(ranked, start=1):
    medal = MEDALS.get(pos, f'  {pos}')
    assert pos == r['expected_pos'], f"Error: {r['name']} debería estar en pos {r['expected_pos']}, está en {pos}"
    print(f"{medal}   {r['candidate']:<22} {r['name']:<25} {float(r['weighted']):>8.2f} {float(r['normalized'])*100:>10.1f}%  {r['resultado']}")

print(f"{'='*70}")

## Ganador seleccionado

In [ ]:
winner = ranked[0]
runner_up = ranked[1]
gap = winner['weighted'] - runner_up['weighted']

print(f"\n🏆 CANDIDATO SELECCIONADO PARA ENTREVISTA FINAL")
print(f"{'='*50}")
print(f"  Nombre:    {winner['candidate']}")
print(f"  Score:     {winner['weighted']} / 5.00  ({float(winner['normalized'])*100:.1f}%)")
print(f"  Resultado: {winner['resultado']}")
print(f"  Ventaja:   +{gap:.2f} puntos sobre {runner_up['candidate']}")
print(f"{'='*50}")

# Validación final
assert winner['name'] == 'ganador_claro', f"El ganador debería ser 'ganador_claro', es '{winner['name']}'"
print("\n✓ El sistema seleccionó al candidato correcto")
print("✓ El ranking reproduce el orden esperado exactamente")

## Visualización — Scores por dimensión

In [ ]:
try:
    import matplotlib.pyplot as plt
    import numpy as np

    dim_labels = [f"D{i+1}\n(×{PESOS[f'D{i+1}']})"
                  for i in range(8)]
    
    colors = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c']
    labels_map = {
        'ganador_claro': 'Ganador claro',
        'buen_candidato': 'Buen candidato',
        'candidato_promedio': 'Candidato promedio',
        'respuestas_debiles': 'Respuestas débiles',
    }

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

    # Gráfico 1: scores por dimensión (radar-like bar)
    x = np.arange(8)
    width = 0.2
    for i, name in enumerate(SCENARIOS):
        s = scenarios[name]
        dim_scores = [a['simulated_ai_score'] for a in s['answers']]
        ax1.bar(x + i * width, dim_scores, width, label=labels_map[name], color=colors[i], alpha=0.85)

    ax1.set_xlabel('Dimensión')
    ax1.set_ylabel('Score (1–5)')
    ax1.set_title('Scores por dimensión — los 4 escenarios')
    ax1.set_xticks(x + width * 1.5)
    ax1.set_xticklabels(dim_labels)
    ax1.set_ylim(0, 5.5)
    ax1.axhline(y=3, color='gray', linestyle='--', alpha=0.5, label='Mínimo aceptable')
    ax1.legend(loc='lower right', fontsize=8)

    # Gráfico 2: score final ponderado
    scenario_labels = [labels_map[n] for n in SCENARIOS]
    final_scores = [float(r['weighted']) for r in results]
    bars = ax2.barh(scenario_labels, final_scores, color=colors, alpha=0.85)
    ax2.set_xlabel('Score ponderado final (1–5)')
    ax2.set_title('Ranking final — Score ponderado')
    ax2.set_xlim(0, 5.5)
    ax2.axvline(x=3, color='gray', linestyle='--', alpha=0.5)
    for bar, score in zip(bars, final_scores):
        ax2.text(score + 0.05, bar.get_y() + bar.get_height()/2,
                 f'{score:.2f}', va='center', fontweight='bold')

    plt.tight_layout()
    plt.savefig(ROOT / 'notebooks' / 'interview_scores.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✓ Gráfico guardado en notebooks/interview_scores.png')

except ImportError:
    print('matplotlib no instalado. Instala con: uv add matplotlib')
    print('Los scores finales son:', {r['name']: float(r['weighted']) for r in results})

## Resumen del sistema

| Componente | Rol |
|---|---|
| `InterviewConductor` | Gestiona el flujo de 8 preguntas, guarda respuestas en `context_summary` |
| `AIClient` (gpt-4o-mini) | Evalúa cada respuesta contra la rúbrica — 1 llamada por dimensión |
| `Scorer` | Calcula `Σ(score × peso) / 19` — 100% Python, sin IA |
| `SessionManager` | Mutex Redis para evitar mensajes concurrentes al mismo candidato |
| `evaluations` table | Almacena el resultado final; `dimension_scores` guarda los 8 sub-scores |

**Principio clave:** La IA solo puntúa texto. El ranking, la decisión APTO/DESCARTADO y los pesos son siempre código Python.